# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library and pandas.

### Dataset Source
The dataset source is described via a [Croissant schema](https://mlcommons.org/croissant/) with the following URL:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install mlcroissant if not already installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata # Don't treat metadata as a dict; it's an object

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review record sets and fields available in the dataset using their `@id` values.

In [ ]:
# List available record sets by their @id and fields
print("Available record sets and their fields:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- RecordSet name: {getattr(rs, 'name', '')}  @id: {rs.id}")
    # List fields with @id for each record set
    if hasattr(rs, 'fields'):
        for f in rs.fields:
            print(f"    - Field: {getattr(f, 'name', '')}   @id: {f.id}")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for further analysis. Make sure to use `@id` values to reference record sets and fields.

In [ ]:
# Compile RecordSet @ids for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
dfs = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dfs[record_set_id] = df
        print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns for RecordSet @id: {record_set_id}")
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

# Show the columns of the main record set (choose the largest DataFrame or as appropriate)
if dfs:
    # Choose the main data table (largest DataFrame)
    main_record_set_id = max(dfs, key=lambda k: dfs[k].shape[0])
    print(f"\nMain record set selected: {main_record_set_id}")
    print("Fields (@id):", list(dfs[main_record_set_id].columns))
    display(dfs[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's conduct exploratory analysis by selecting a numeric field (with its `@id`) from the main record set and applying some basic filtering, normalization, and group-wise statistics. Please update the `numeric_field_id` and `group_field_id` according to your dataset's fields. The code below demonstrates typical EDA steps on the chosen record set.

In [ ]:
# --- EDA Section ---
# Manually specify @id of numeric field and group field by checking above output.
# For this dataset (see Data Overview above), we'll choose sample plausible field ids.

# Replace these with actual @id values from your dataset fields
# Example placeholder values (replace after checking the output of the above cell):
numeric_field_id = None  # e.g. 'http://senscience.ai/Age@recordSet01' (replace!)
group_field_id = None    # e.g. 'http://senscience.ai/Sex@recordSet01' (replace!)

# Define main dataframe
main_df = dfs[main_record_set_id]

# Attempt automatic selection if not set manually
if numeric_field_id is None:
    # Try to pick a float/integer column by heuristics
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break
if group_field_id is None:
    # Use first non-numeric/few-unique column by heuristic
    for col in main_df.columns:
        if not pd.api.types.is_numeric_dtype(main_df[col]) and main_df[col].nunique() < 10:
            group_field_id = col
            break
print(f"Numeric field selected (@id): {numeric_field_id}")
print(f"Group field selected (@id): {group_field_id}")

# Filter for records where numeric_field > threshold (choose threshold based on data)
if numeric_field_id and numeric_field_id in main_df.columns:
    threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field and compute mean
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name='mean')
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization
Plot distributions and relationships for the chosen numeric and group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if data is available for visualization
if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=15, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Show group comparison if available
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=main_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to access and explore the FAIR² dataset, referencing all entities by their `@id`. We demonstrated loading of the dataset, overviewing metadata, extracting record sets and fields, basic statistical analysis, and visualization.

The approach shown here can be adapted for any dataset described with the Croissant schema. Please consult the dataset's documentation for specific details about field types and semantics for more advanced analyses.